# MVP RLSSM Model — 4-Arm Drifting Bandit

This notebook builds a **choice-only RLSSM** for the MindRL Challenge 4-arm
drifting bandit, fits it with HSSM, and evaluates one-step-ahead predictions.

**Model:** Rescorla-Wagner learning + inverse-temperature softmax (choice-only)

$$Q_c \leftarrow Q_c + \alpha \,(r - Q_c)$$

$$p(\text{choose } a) = \frac{\exp(\beta \cdot Q_a)}{\sum_k \exp(\beta \cdot Q_k)}$$

**Parameters:**
- `rl_alpha` — learning rate (how quickly Q-values update from feedback)
- `beta` — inverse temperature (how strongly Q-values determine choices)

**Generalisation:** The same `rl_alpha` and `beta` are N-agnostic — only the
number of Q-values and the decision-process name (`inv_temp_softmax_N`)
change for different arm counts.  The learner in `bayesd_misfits.model`
is parameterised by `n_actions`, so swapping to 2/3/N arms requires only
changing that constructor argument and the `decision_process` string.

## 1. Setup

In [ ]:
import logging
import os
import warnings

import arviz as az
import numpy as np
import pandas as pd

import hssm
from ssms import rl
from ssms.rl import ModelConfig
from ssms.rl.env import Bandit

from bayesd_misfits.data import ensure_data_downloaded, load_challenge_data, summarize_data
from bayesd_misfits.model import NArmRescorlaWagner

warnings.filterwarnings("ignore")
logging.getLogger("jax._src.xla_bridge").setLevel("ERROR")
hssm.set_floatX("float32", update_jax=True)

RANDOM_SEED = 20260719

In [ ]:
# Scale controls — keep small for quick iteration, increase for real fits.
FULL_RUN = os.environ.get("FULL_RUN", "0") == "1"

N_PARTICIPANTS = 50 if FULL_RUN else 20
N_TRIALS = 120                      # fixed trial count (balanced panel requirement)
N_CHAINS = 4 if FULL_RUN else 2
N_TUNE = 1000 if FULL_RUN else 500
N_DRAWS = 1000 if FULL_RUN else 500

print(f"FULL_RUN={FULL_RUN} | participants={N_PARTICIPANTS} trials={N_TRIALS} "
      f"tune={N_TUNE} draws={N_DRAWS} chains={N_CHAINS}")

## 2. Load data

HSSM requires **balanced panels** (same trial count per participant). We filter
to trajectories with exactly 120 trials (the most common length — 1981 of 2678)
and subsample participants for the MVP.

In [ ]:
ensure_data_downloaded()

df = load_challenge_data(
    feedback_transform="normalize",   # reward / 100 → [0, 1]
    rt_placeholder=-1.0,              # choice-only (no RT modelling)
    group_by="trajectory",            # each trajectory = one participant
)

# Keep only trajectories with exactly N_TRIALS trials (balanced panel)
trial_counts = df.groupby("participant_id").size()
valid_pids = trial_counts[trial_counts == N_TRIALS].index
df = df[df["participant_id"].isin(valid_pids)].reset_index(drop=True)

# Subsample participants
rng = np.random.default_rng(RANDOM_SEED)
all_pids = sorted(df["participant_id"].unique())
selected_pids = rng.choice(all_pids, size=min(N_PARTICIPANTS, len(all_pids)), replace=False)
df = df[df["participant_id"].isin(selected_pids)].sort_values(["participant_id", "trial_id"]).reset_index(drop=True)

# Remap participant_id to 0..N-1 (HSSM expects contiguous IDs)
pid_map = {pid: i for i, pid in enumerate(sorted(df["participant_id"].unique()))}
df["participant_id"] = df["participant_id"].map(pid_map)

print(f"Selected {df['participant_id'].nunique()} participants × {N_TRIALS} trials = {len(df)} rows")
summarize_data(df)

In [ ]:
# Choice-only data — drop rt, keep only what HSSM needs
hssm_data = df[["participant_id", "trial_id", "response", "feedback"]].copy()
hssm_data.head(10)

## 3. Build the custom 4-arm RLSSM model

Three components (following the Carney ARIA Workshop custom-model pattern):

1. **Task environment** — `Bandit.bernoulli` with 4 arms (response labels 0–3).
2. **Learning process** — `NArmRescorlaWagner` (our custom learner in
   `bayesd_misfits.model`).  Maintains 4 Q-values, computes `q0..q3` each trial,
   updates via the RW delta rule.
3. **Decision process** — `inv_temp_softmax_4` (registered in HSSM).  Computes
   choice probability as softmax of `beta * Q` over 4 arms.

In [ ]:
# ── 3a. Learning process: 4-arm Rescorla-Wagner ──
# initial_q=0.5 because feedback is normalised to [0, 1]
learner = NArmRescorlaWagner(
    n_actions=4,
    initial_q=0.5,
    use_decay=False,          # set True to add forgetting for restless bandits
)

# ── 3b. Task environment: 4-arm bandit (dummy probabilities — only used for simulation) ──
env = Bandit.bernoulli(
    probabilities=[0.25, 0.25, 0.25, 0.25],
    response_labels=[0, 1, 2, 3],
)

# ── 3c. Assemble the ModelConfig ──
ssms_config = ModelConfig(
    model_name="4AB_RW_Softmax",
    description="4-arm bandit: Rescorla-Wagner + inv-temp softmax (choice-only)",
    decision_process="inv_temp_softmax_4",
    learning_process=learner,
    task_environment=env,
    response=["response"],       # choice-only — no rt column
)

ssms_config.validate()
assembled = ssms_config.assemble(backend="jax")

print("Model config:")
print(f"  model_name:        {ssms_config.model_name}")
print(f"  decision_process:  {ssms_config.decision_process}")
print(f"  list_params:       {ssms_config.list_params}")
print(f"  choices:           {ssms_config.choices}")
print(f"  context_fields:    {ssms_config.context_fields}")
print(f"  bounds:            {ssms_config.bounds}")
print(f"  computed_params:   {assembled.computed_params}")
print(f"  gradient:          {assembled.gradient}")

In [ ]:
# Validate our real data against the model contract
ssms_config.validate_data(hssm_data).raise_for_errors()
print("Real data validation passed ✓")

## 4. Bridge to HSSM and specify hierarchical priors

Each parameter gets a **group intercept** + **per-participant deviation**:

```
rl_alpha ~ 1 + (1 | participant_id)
beta     ~ 1 + (1 | participant_id)
```

In [ ]:
# Bridge ssms model → HSSM config
model_config = hssm.rl.RLSSMConfig.from_ssms_model(ssms_config)

print(f"list_params:   {model_config.list_params}")
print(f"extra_fields:  {model_config.extra_fields}")
print(f"is_choice_only: {model_config.is_choice_only}")
print(f"computed:       {set(model_config.ssm_logp_func.computed)}")

In [ ]:
PARTICIPANT_EFFECT_PRIOR = {
    "name": "Normal",
    "mu": 0,
    "sigma": {"name": "HalfNormal", "sigma": 0.5},
}


def hierarchical_param(name, lower, upper, mu, sigma):
    """Group intercept (TruncatedNormal) + per-participant random effect."""
    return hssm.Param(
        name,
        formula=f"{name} ~ 1 + (1|participant_id)",
        prior={
            "Intercept": hssm.Prior(
                "TruncatedNormal", lower=lower, upper=upper, mu=mu, sigma=sigma
            ),
            "1|participant_id": PARTICIPANT_EFFECT_PRIOR,
        },
    )


model = hssm.RLSSM(
    data=hssm_data,
    model_config=model_config,
    p_outlier=0,
    lapse=None,
    process_initvals=False,   # critical for RLSSM — start from prior
    include=[
        hierarchical_param("rl_alpha", 0.0, 1.0, mu=0.2, sigma=0.15),
        hierarchical_param("beta", 0.0, 10.0, mu=3.0, sigma=1.5),
    ],
)

print(f"participants: {model.n_participants} | trials/participant: {model.n_trials}")
print(f"free params: {list(model.params.keys())}")

## 5. Sample the posterior

In [ ]:
idata = model.sample(
    sampler="numpyro",
    draws=N_DRAWS,
    tune=N_TUNE,
    chains=N_CHAINS,
    cores=1,
    target_accept=0.9,
    random_seed=RANDOM_SEED,
    idata_kwargs={"log_likelihood": False},
)
idata

## 6. Check convergence and parameter estimates

In [ ]:
# Group-level intercepts
az.summary(
    idata,
    var_names=["rl_alpha_Intercept", "beta_Intercept"],
    kind="all",
    round_to=3,
)

In [ ]:
# Convergence diagnostics
max_rhat = max(float(az.rhat(idata)[v].max()) for v in az.rhat(idata).data_vars)
divergences = int(idata.sample_stats["diverging"].sum())
print(f"Max R-hat:       {max_rhat:.3f}  (target < 1.01)")
print(f"Divergences:     {divergences}")

# Per-participant posterior means
post = idata.posterior
if hasattr(post, "to_dataset"):
    post = post.to_dataset()
post = post.stack(sample=("chain", "draw"))

for name in ["rl_alpha", "beta"]:
    re = post[f"{name}_1|participant_id"]
    pid_dim = [d for d in re.dims if d != "sample"][0]
    draws = post[f"{name}_Intercept"] + re
    means = draws.mean("sample").values
    print(f"\n{name}: per-participant posterior means")
    print(f"  range: [{means.min():.3f}, {means.max():.3f}], mean: {means.mean():.3f}")

## 7. Posterior predictive check (RLSSM-aware)

`mode="ppc"` replays each participant's **observed** responses and feedback to
keep Q-values on the same learning path, and only simulates fresh choices from
posterior parameters. This isolates decision fit from bandit randomness.

In [ ]:
LIST_PARAMS = ["rl_alpha", "beta"]


def draw_posterior_theta(idata, draw_idx):
    """Return a single posterior draw of per-participant parameters."""
    posterior = idata.posterior
    if hasattr(posterior, "to_dataset"):
        posterior = posterior.to_dataset()
    post = posterior.stack(sample=("chain", "draw"))
    theta = {}
    for name in LIST_PARAMS:
        re = post[f"{name}_1|participant_id"]
        pid_dim = [d for d in re.dims if d != "sample"][0]
        vals = (post[f"{name}_Intercept"] + re).isel(sample=draw_idx)
        ids = [int(v) for v in re[pid_dim].values]
        s = pd.Series(np.asarray(vals.values), index=ids).sort_index()
        theta[name] = s.reindex(range(model.n_participants)).to_numpy()
    return theta


N_PPC_DRAWS = 20 if FULL_RUN else 8
n_samples = idata.posterior.sizes["chain"] * idata.posterior.sizes["draw"]
ppc_rng = np.random.default_rng(RANDOM_SEED + 1)
draw_ids = ppc_rng.choice(n_samples, size=min(N_PPC_DRAWS, n_samples), replace=False)

ppc_frames = []
for k, d in enumerate(draw_ids):
    theta_d = draw_posterior_theta(idata, int(d))
    ppc_d = rl.Simulator(ssms_config).simulate(
        theta=theta_d,
        mode="ppc",
        observed_data=hssm_data,
        random_state=RANDOM_SEED + 100 + k,
    )
    ppc_d["ppc_draw"] = k
    ppc_frames.append(ppc_d)

ppc_data = pd.concat(ppc_frames, ignore_index=True)
print(f"PPC draws: {len(draw_ids)} | total rows: {len(ppc_data)}")

In [ ]:
# Compare action distributions: observed vs posterior-predictive
choices = [0, 1, 2, 3]
obs_props = hssm_data["response"].value_counts(normalize=True).reindex(choices, fill_value=0.0)

ppc_props = pd.DataFrame(
    [g["response"].value_counts(normalize=True).reindex(choices, fill_value=0.0)
     for _, g in ppc_data.groupby("ppc_draw")]
)

print("Action proportions:")
print(f"  {'Arm':>6}  {'Observed':>9}  {'PPC mean':>9}  {'PPC 3%':>8}  {'PPC 97%':>8}")
for c in choices:
    print(f"  {c:>6}  {obs_props[c]:>9.3f}  {ppc_props[c].mean():>9.3f}  "
          f"{ppc_props[c].quantile(0.03):>8.3f}  {ppc_props[c].quantile(0.97):>8.3f}")

In [ ]:
# Learning-curve PPC: P(choose best-recent arm) over trial bins
BIN = 10

def learning_curve(df, bin_size=BIN):
    """Mean chosen-arm reward over trial bins (proxy for learning)."""
    d = df.copy()
    d["trial_bin"] = (d["trial_id"] // bin_size) * bin_size
    return d.groupby("trial_bin")["feedback"].mean()

obs_curve = learning_curve(hssm_data)
ppc_curves = pd.concat(
    [learning_curve(g).rename(k) for k, g in ppc_data.groupby("ppc_draw")],
    axis=1,
).sort_index()

centers = obs_curve.index + BIN / 2
print(f"Observed mean feedback (overall): {hssm_data['feedback'].mean():.3f}")
print(f"PPC mean feedback (overall):      {ppc_data['feedback'].mean():.3f}")

## 8. One-step-ahead prediction (challenge metric)

The challenge evaluates **one-step-ahead prediction** using negative
log-likelihood (NLL) or cross-entropy. For each trial, given the history of
observed actions and rewards, the model predicts the probability of the next
action. Lower NLL = better prediction.

We compute NLL by replaying each participant's trial sequence with posterior
parameter draws and evaluating the softmax choice probability against the
actual observed response.

In [ ]:
from scipy.special import logsumexp


def compute_one_step_nll(idata, data, config, n_draws=200):
    """Compute one-step-ahead NLL using posterior parameter draws.

    For each posterior draw, replays each participant's trial sequence,
    updates Q-values from observed (action, reward) pairs, and evaluates
    the log-probability of the observed action under the softmax.
    """
    posterior = idata.posterior
    if hasattr(posterior, "to_dataset"):
        posterior = posterior.to_dataset()
    post = posterior.stack(sample=("chain", "draw"))

    n_total = post.sizes["sample"]
    draw_indices = np.random.default_rng(42).choice(
        n_total, size=min(n_draws, n_total), replace=False
    )

    n_participants = data["participant_id"].nunique()
    n_trials = data.groupby("participant_id").size().iloc[0]
    initial_q = 0.5

    all_nlls = []

    for d_idx in draw_indices:
        # Get per-participant parameters for this draw
        theta = draw_posterior_theta(idata, int(d_idx))
        alphas = theta["rl_alpha"]
        betas = theta["beta"]

        for pid in range(n_participants):
            alpha = alphas[pid]
            beta = betas[pid]
            Q = np.full(4, initial_q)

            pid_data = data[data["participant_id"] == pid].sort_values("trial_id")

            for _, row in pid_data.iterrows():
                action = int(row["response"])
                reward = float(row["feedback"])

                # One-step-ahead prediction: log P(observed action | Q)
                logits = beta * Q
                logp = logits[action] - logsumexp(logits)
                all_nlls.append(-logp)

                # Update Q-value with observed outcome
                Q[action] += alpha * (reward - Q[action])

    nlls = np.array(all_nlls)
    # Average over draws, then over trials
    nll_per_draw = nlls.reshape(len(draw_indices), -1).mean(axis=1)
    return nll_per_draw


nlls = compute_one_step_nll(idata, hssm_data, ssms_config, n_draws=100)
print(f"One-step-ahead NLL (per trial):")
print(f"  mean:   {nlls.mean():.4f}")
print(f"  std:    {nlls.std():.4f}")
print(f"  94% HDI: [{np.quantile(nlls, 0.03):.4f}, {np.quantile(nlls, 0.97):.4f}]")
print(f"\nFor reference:")
print(f"  Uniform random (4 arms):  ln(4) = {np.log(4):.4f}")
print(f"  Perfect prediction:       0.0000")

## 9. Generalisation to N-arm tasks

The model is designed to generalise to **any** N-arm bandit. The cognitive
parameters `rl_alpha` and `beta` have the same meaning regardless of N — only
the number of Q-values and the decision-process name change.

To adapt for a heldout N-arm task:

```python
# For a 3-arm task:
learner_3 = NArmRescorlaWagner(n_actions=3, initial_q=0.5)
env_3 = Bandit.bernoulli(probabilities=[0.4, 0.3, 0.3], response_labels=[0, 1, 2])
config_3 = ModelConfig(
    model_name="3AB_RW_Softmax",
    decision_process="inv_temp_softmax_3",   # HSSM has 2, 3, 4 registered
    learning_process=learner_3,
    task_environment=env_3,
    response=["response"],
)
```

For N > 4 (e.g. 6-arm), register a custom softmax in HSSM:

```python
from hssm.rl.registry import register_ssm
# register_ssm can create inv_temp_softmax_N for any N
# using the same _make_inv_temp_softmax_base_logp factory pattern
```

The fitted `rl_alpha` and `beta` posteriors transfer directly — the learning
rate governs how fast Q-values update (independent of arm count), and the
inverse temperature controls exploration vs exploitation (also arm-agnostic,
though the effective choice concentration depends on N).

## 10. Summary

We built and fit a **choice-only RLSSM** for the MindRL 4-arm drifting bandit:

1. **Custom learner** (`NArmRescorlaWagner`) — 4-arm RW delta rule with JAX gradients
2. **Decision process** (`inv_temp_softmax_4`) — inverse-temperature softmax over 4 Q-values
3. **Hierarchical priors** — group intercept + per-participant deviation for `rl_alpha` and `beta`
4. **Posterior sampling** — NumPyro NUTS with `process_initvals=False`
5. **Posterior predictive checks** — RLSSM-aware PPC via `mode="ppc"`
6. **One-step-ahead NLL** — the challenge evaluation metric

### Next steps

- **Add forgetting** (`use_decay=True`) — a `rl_decay` parameter that pulls Q-values
  toward the initial value each trial, helping the model track drifting rewards.
- **RT-based model** — swap `inv_temp_softmax_4` for `race_no_bias_angle_4` to model
  response times alongside choices (requires the `2AB_RW_Angle`-style drift computation).
- **Full-scale fit** — set `FULL_RUN=1` for more participants, draws, and chains.
- **Model comparison** — compare RW vs. dual-alpha (positive/negative learning rates)
  vs. decay-augmented RW using NLL or WAIC/LOO.